In [ ]:
import pandas as pd
import numpy as np

import urllib3
from bs4 import BeautifulSoup
import time

def soupify(url: str):

    http = urllib3.PoolManager()
    response = http.request('GET', url)
    soup = BeautifulSoup(response.data)

    return soup

# fixing matchups and building db

In [ ]:
url: str = "https://www.sports-reference.com/cbb/postseason/men/2026-ncaa.html"
s = soupify(url)

In [ ]:
matchups = pd.DataFrame(columns = ["season", "round", "team1", "team2", "team1_seed", "team2_seed", "win"])
for year in range(2010, 2027):
    if year != 2020:
        s = soupify(f"https://www.sports-reference.com/cbb/postseason/men/{year}-ncaa.html")
        for div in list(d for d in s.find('div', {"id" : "brackets"}).find_all('div') if d.get('id') and d.get('id') == "bracket"):
            seeds = list(d.contents[0] for d in div.find_all('span') if not d.find('a'))[:-1]
            seeds = list(c for c in seeds if c.isnumeric())
            teams = list(a.contents[0] for a in div.find_all('a') if "boxscores" not in a.get('href'))[:-1]
            winners = list(d.find('a').contents[0] for d in div.find_all('div', {"class" : "winner"}))
            if len(winners) == 14:
                winners.insert(6, "Oregon")
            round = 64 if len(winners) > 3 else 4
            for i in range(len(winners)):
                matchups.loc[len(matchups.index)] = [
                    year, int(round), teams[2*i], teams[2*i + 1], seeds[2*i], seeds[2*i + 1],
                    1 if winners[i] == teams[2*i] else 0
                ]
                if round == 4:
                    round /= 2 if i == 1 else 1
                if i == 7 or i == 11 or i == 13:
                    round /= 2
        time.sleep(5)

In [ ]:
matchups.loc[matchups['season'] == 2021,].iloc[45:]

,season,round,team1,team2,team1_seed,team2_seed,win
675,2021,64,Gonzaga,Norfolk State,1,16,1
676,2021,64,Oklahoma,Missouri,8,9,1
677,2021,64,Creighton,UC Santa Barbara,5,12,1
678,2021,64,Virginia,Ohio,4,13,0
679,2021,64,USC,Drake,6,11,1
680,2021,64,Kansas,Eastern Washington,3,14,1
681,2021,64,Oregon,VCU,7,10,0
682,2021,64,Iowa,Grand Canyon,"at Indianapolis, IN",2,0
683,2021,32,Gonzaga,Oklahoma,15,1,0
684,2021,32,Creighton,Ohio,8,5,0


In [ ]:
pd.concat([matchups, pd.read_csv("matchups1025.csv")]).to_csv("matchups1026.csv", index = False)

In [ ]:
matchups = pd.read_csv("matchups1025.csv")
matchups

,season,round,team1,team2,team1_seed,team2_seed,win
0,2025,64,Duke,Mount St. Mary's,1,16,1
1,2025,64,Mississippi State,Baylor,8,9,0
2,2025,64,Oregon,Liberty,5,12,1
3,2025,64,Arizona,Akron,4,13,1
4,2025,64,BYU,VCU,6,11,1
...,...,...,...,...,...,...,...
940,2010,16,Xavier,Kansas State,6,2,0
941,2010,8,Butler,Kansas State,5,2,1
942,2010,4,Duke,West Virginia,1,2,1
943,2010,4,Michigan State,Butler,5,5,0


In [ ]:
matchups = pd.concat([matchups, pd.read_csv("matchups1025.csv")])

In [ ]:
SRS = pd.read_csv("SRS_SOS_fixed_abbr.csv")
stats = pd.read_csv("teamstats_10-26_d1_FIXED.csv")

In [ ]:
# temp = pd.merge(stats.drop(["SRS","SOS"], axis = 1), SRS[["year","team","SRS","SOS"]], left_on=["year","team"], right_on = ["year","team"]).copy()
df = pd.merge(SRS[["year","abbr","SRS","SOS"]], stats.drop(["SRS","SOS"], axis = 1), right_on=["year","team"], left_on = ["year","abbr"]).copy()
# temp = temp.drop("team", axis = 1)
# df = temp.rename(columns = {"abbr" : "team"})

In [ ]:
# stats.loc[stats["team"].str.contains("Mc"),]

In [ ]:
pd.merge(stats.loc[stats["team"].str.contains("Mc")],
         SRS, left_on = ["year", "team"], right_on= ["year","team"])

,year,team,G,MP,pts,opp_pts,fg,fga,fg3,fg3a,...,opp_pf_100poss,3par,drb_pct,trb_pct,ftr,MoV,Rating,SOS_y,SRS_y,abbr
0,2024,McNeese,29,1160,2275,1857,796,1648,218,554,...,27.602837,0.336165,0.713823,0.510801,0.405947,14.413793,7.528049,-6.885744,7.528049,McNeese State
1,2025,McNeese,31,1240,2335,1987,828,1772,232,651,...,24.078254,0.367381,0.707194,0.522347,0.356659,11.225806,9.252967,-1.972839,9.252967,McNeese State
2,2026,McNeese,31,1255,2382,2078,830,1828,203,642,...,25.944206,0.351204,0.683084,0.509406,0.383479,9.806452,8.160550,-1.645902,8.160550,McNeese State


In [ ]:
# stats.loc[stats["team"].str.contains("Mc")]
temp = pd.merge(SRS[["year","abbr","SRS","SOS"]], stats.drop(["SRS","SOS"], axis = 1), left_on = ["year","abbr"], right_on=["year","team"]).copy()
temp2 = pd.merge(SRS[["year","team","SRS","SOS"]], stats.drop(["SRS","SOS"], axis = 1), left_on = ["year","team"], right_on=["year","team"]).copy()
temp = pd.concat([temp, temp2]).groupby(["year","team"]).head(1).copy()
df = temp.drop("abbr", axis = 1)

In [ ]:
df.to_csv("teamstats_10-26_d1_FIXED.csv", index = False)

In [ ]:
matchups = pd.read_csv("matchups1026.csv")

In [ ]:
# matchups['team1_seed'] = pd.to_numeric(matchups['team1_seed'])
# matchups['team2_seed'] = pd.to_numeric(matchups['team2_seed'])

# matchups.to_csv("matchups1026.csv", index = False)

In [ ]:
# df = df.rename(columns = {"abbr" : "team", "team" : "abbr"})
temp = pd.merge(matchups, df.add_suffix("_team1"), left_on=["season","team1"], right_on = ["year_team1","team_team1"]).copy()
temp = temp.drop(["year_team1", "team_team1"], axis = 1)
temp = pd.merge(temp, df.add_suffix("_team2"), left_on=["season","team2"], right_on = ["year_team2","team_team2"]).copy()
temp = temp.drop(["year_team2", "team_team2"], axis = 1)

In [ ]:
list(c for c in matchups['team2'].values if c not in temp["team"].values)

[]

In [ ]:
temp

,season,round,team1,team2,team1_seed,team2_seed,win,SRS_team1,SOS_team1,G_team1,...,opp_blk_G_team2,opp_blk_100poss_team2,opp_tov_G_team2,opp_tov_100poss_team2,opp_pf_G_team2,opp_pf_100poss_team2,3par_team2,drb_pct_team2,trb_pct_team2,ftr_team2
0,2010,64,Kentucky,ETSU,1,16,1,21.305654,7.335066,34,...,3.545455,5.130735,16.727273,24.206545,18.848485,27.276216,0.289092,0.621801,0.504598,0.400322
1,2010,64,Texas,Wake Forest,8,9,0,19.759047,8.213592,33,...,3.827586,5.318321,14.137931,19.644247,20.896552,29.035156,0.231700,0.639680,0.526871,0.410375
2,2010,64,Temple,Cornell,5,12,0,13.645673,4.706279,34,...,2.793103,4.232003,13.793103,20.898778,16.793103,25.444262,0.399263,0.673188,0.512642,0.288698
3,2010,64,Wisconsin,Wofford,4,13,1,20.114138,8.823815,31,...,3.969697,6.045573,14.212121,21.644073,19.575758,29.812518,0.311901,0.674627,0.517922,0.445572
4,2010,64,Marquette,Washington,6,11,0,16.388160,9.162353,33,...,3.090909,4.217861,16.090909,21.957687,21.303030,29.070158,0.258477,0.651696,0.532521,0.420147
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,2026,16,Texas,Purdue,11,2,0,16.585834,10.714866,31,...,2.342857,3.528722,9.942857,14.975552,16.914286,25.475651,0.409265,0.752354,0.551152,0.286055
1004,2026,8,Arizona,Purdue,1,2,1,30.336976,12.984034,34,...,2.342857,3.528722,9.942857,14.975552,16.914286,25.475651,0.409265,0.752354,0.551152,0.286055
1005,2026,4,UConn,Illinois,2,3,1,23.331744,10.949391,34,...,2.593750,3.811492,7.281250,10.699730,18.718750,27.507032,0.507056,0.741379,0.562573,0.329133
1006,2026,4,Michigan,Arizona,1,1,1,32.893631,15.275984,34,...,3.705882,5.086208,11.823529,16.227426,19.676471,27.005344,0.268246,0.754732,0.567704,0.428710


In [ ]:
temp.to_csv("database10-26.csv", index = False)

# training

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.ensemble import RandomForestClassifier

In [ ]:
db = pd.read_csv("database10-26.csv")
db.head()

,season,round,team1,team2,team1_seed,team2_seed,win,SRS_team1,SOS_team1,G_team1,...,opp_blk_G_team2,opp_blk_100poss_team2,opp_tov_G_team2,opp_tov_100poss_team2,opp_pf_G_team2,opp_pf_100poss_team2,3par_team2,drb_pct_team2,trb_pct_team2,ftr_team2
0,2010,64,Kentucky,ETSU,1,16,1,21.305654,7.335066,34,...,3.545455,5.130735,16.727273,24.206545,18.848485,27.276216,0.289092,0.621801,0.504598,0.400322
1,2010,64,Texas,Wake Forest,8,9,0,19.759047,8.213592,33,...,3.827586,5.318321,14.137931,19.644247,20.896552,29.035156,0.231700,0.639680,0.526871,0.410375
2,2010,64,Temple,Cornell,5,12,0,13.645673,4.706279,34,...,2.793103,4.232003,13.793103,20.898778,16.793103,25.444262,0.399263,0.673188,0.512642,0.288698
3,2010,64,Wisconsin,Wofford,4,13,1,20.114138,8.823815,31,...,3.969697,6.045573,14.212121,21.644073,19.575758,29.812518,0.311901,0.674627,0.517922,0.445572
4,2010,64,Marquette,Washington,6,11,0,16.388160,9.162353,33,...,3.090909,4.217861,16.090909,21.957687,21.303030,29.070158,0.258477,0.651696,0.532521,0.420147


In [ ]:
db_used = db.loc[:,list(c for c in db.columns if any(t in c for t in {"seed","SRS","SOS","off_rtg","def_rtg","pace","efg_pct","orb_pct","tov_pct","ft_fga", "3par"}) and "opp" not in c)].copy()
comps = list(c.split('_team1')[0] for c in db_used.columns if "_team1" in c)
db_used["seed_diff"] = db_used["team1_seed"] - db_used["team2_seed"]
for comp in comps:
    db_used[f"{comp}_diff"] = db_used[f"{comp}_team1"] - db_used[f"{comp}_team2"]
db_used = db_used.loc[:,list(c for c in db_used.columns if "diff" in c)].copy()

# adding net stats
db_used["net_efg_pct"] = db["efg_pct_team1"] - db["opp_efg_pct_team2"]
db_used["net_tov_pct"] = db["tov_pct_team1"] - db["opp_tov_pct_team2"]

mirrored = -db_used.iloc[:,:-2].copy()
mirrored["net_efg_pct"] = db["efg_pct_team2"] - db["opp_efg_pct_team1"]
mirrored["net_tov_pct"] = db["tov_pct_team2"] - db["opp_tov_pct_team1"]

X = np.vstack([np.array(db_used), np.array(mirrored)])
y = np.array(db["win"])
# y = np.array([y, 1- y])
y = np.hstack([y, 1-y])

In [ ]:
db_used

,seed_diff,SRS_diff,SOS_diff,off_rtg_diff,def_rtg_diff,pace_diff,efg_pct_diff,orb_pct_diff,ft_fga_diff,tov_pct_diff,3par_diff,net_efg_pct,net_tov_pct
0,-15,21.293042,8.977626,12.017826,-3.899379,1.186491,0.050349,0.064006,0.030360,-0.885635,0.010204,0.048903,-3.557902
1,-1,6.858331,-0.865695,8.111010,-1.172802,3.188373,0.039052,0.042419,-0.014311,-1.944941,0.027754,0.083369,-0.902914
2,-7,5.627446,6.910275,-10.007125,-7.983668,-3.382915,-0.075258,0.001678,-0.020297,-1.821543,-0.088152,0.012324,-3.914982
3,-9,15.473520,11.602553,6.485906,-1.296348,-5.437094,0.024008,-0.027515,-0.062655,-2.885886,0.086399,0.048689,-6.133356
4,-5,-0.203936,1.257757,2.861053,2.884117,-8.703040,0.030639,-0.073988,-0.046271,-1.059112,0.092257,0.062396,-5.287983
...,...,...,...,...,...,...,...,...,...,...,...,...,...
1003,9,-9.468961,-3.825643,-5.589124,3.413381,3.675549,-0.033968,-0.007338,0.135168,1.554535,-0.047610,0.019350,-0.773648
1004,-1,4.282181,-1.556475,-4.754713,-11.228707,6.436854,-0.025663,0.030726,0.102133,1.434951,-0.141019,0.027655,-0.893232
1005,-1,-3.535030,-1.354883,-9.362022,-6.276663,0.113672,0.002755,-0.046430,-0.044933,2.681775,-0.103847,0.077126,3.860285
1006,0,2.556656,2.291950,0.748325,0.421571,0.109890,0.030138,-0.035628,-0.033120,1.563221,0.149643,0.130434,-0.167677


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0, train_size=.75)

In [ ]:
rf = RandomForestClassifier(max_depth=4)
rf.fit(X_train, y_train)
rf.score(X_test, y_test)

0.7142857142857143

In [ ]:
rf.feature_importances_

array([0.2650767 , 0.30057596, 0.16365697, 0.07275172, 0.04012675,
       0.01249011, 0.02517264, 0.03986838, 0.01361762, 0.02155428,
       0.01094711, 0.01717816, 0.0169836 ])

In [ ]:
pd.DataFrame({"names" : db_used.columns,
              "importance" : rf.feature_importances_})

,names,importance
0,seed_diff,0.265077
1,SRS_diff,0.300576
2,SOS_diff,0.163657
3,off_rtg_diff,0.072752
4,def_rtg_diff,0.040127
5,pace_diff,0.012490
6,efg_pct_diff,0.025173
7,orb_pct_diff,0.039868
8,ft_fga_diff,0.013618
9,tov_pct_diff,0.021554


In [ ]:
from scipy.stats import uniform

In [ ]:
# cv
params = {
    "criterion" : ["gini", "entropy", "log_loss"],
    "max_depth" : range(1,20),
    "min_samples_split" : list(uniform.rvs() for _ in range(50)),
    "min_samples_leaf" : list(uniform.rvs() for _ in range(50)),
    "max_features" : list(uniform.rvs() for _ in range(50))
}
rf = RandomizedSearchCV(RandomForestClassifier(),
                        param_distributions = params,
                        n_jobs = -1, random_state=0,
                        n_iter = 100, scoring = "neg_log_loss")
rf.fit(X_train, y_train)

RandomizedSearchCV(estimator=RandomForestClassifier(), n_iter=100, n_jobs=-1,
                   param_distributions={'criterion': ['gini', 'entropy',
                                                      'log_loss'],
                                        'max_depth': range(1, 20),
                                        'max_features': [np.float64(0.7970509948960693),
                                                         np.float64(0.7477277802631984),
                                                         np.float64(0.5566313475658103),
                                                         np.float64(0.48292004406475),
                                                         np.float64(0.8965343488046879),
                                                         np.float64(0.889502...
                                                              np.float64(0.7466449894663243),
                                                              np.float64(0.8875865006534529),
                                                              np.float64(0.4732191351488547),
                                                              np.float64(0.49742634210648873),
                                                              np.float64(0.09817490004324114),
                                                              np.float64(0.1475793015505925),
                                                              np.float64(0.5374660434318039),
                                                              np.float64(0.5577733470313004),
                                                              np.float64(0.6021225478409644),
                                                              np.float64(0.265533243566757), ...]},
                   random_state=0, scoring='neg_log_loss')

In [ ]:
rf.best_params_

{'min_samples_split': np.float64(0.22174775858728168),
 'min_samples_leaf': np.float64(0.05982308095151079),
 'max_features': np.float64(0.8895020034307322),
 'max_depth': 8,
 'criterion': 'gini'}

In [ ]:
# rf.score(X_train, y_train)
np.mean(rf.predict(X_train) == y_train)

np.float64(0.7175925925925926)

In [ ]:
# rf.score(X_test, y_test)
np.mean(rf.predict(X_test) == y_test)

np.float64(0.7123015873015873)

In [ ]:
np.exp(rf.score(X_train, y_train))

np.float64(0.5804466145450741)

In [ ]:
from sklearn.metrics import precision_score
precision_score(y_test, rf.predict(X_test))

0.722007722007722

In [ ]:
pd.DataFrame({"names" : db_used.columns,
              "importance" : rf.best_estimator_.feature_importances_})

,names,importance
0,seed_diff,0.114805
1,SRS_diff,0.838315
2,SOS_diff,0.030340
3,off_rtg_diff,0.002386
4,def_rtg_diff,0.000000
5,pace_diff,0.004190
6,efg_pct_diff,0.000975
7,orb_pct_diff,0.003527
8,ft_fga_diff,0.001999
9,tov_pct_diff,0.001730


# writeup

In [1]:
import torch
from sklearn.ensemble import AdaBoostClassifier
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
import pickle

In [2]:
with open('ada_boost_model.pkl', 'rb') as file:
    ada = pickle.load(file)

with open('standard_scaler-2.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [3]:
class Model(torch.nn.Module):
    def __init__(self, input_size: int, output_size: int = 1, hidden_dim: int = 32, n_layers: int = 2, dropout: float= .5):
        super(Model, self).__init__()

        self.layers = torch.nn.ModuleList()
        self.bn_layers = torch.nn.ModuleList() # New list for BatchNorm

        # Input Layer
        # Pro-tip: bias=False because BN has its own 'beta' parameter that handles shifting
        self.layers.append(torch.nn.Linear(input_size, hidden_dim, bias=False))
        self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim))

        # Hidden Layers
        for _ in range(n_layers):
            self.layers.append(torch.nn.Linear(hidden_dim, hidden_dim, bias=False))
            self.bn_layers.append(torch.nn.BatchNorm1d(hidden_dim))

        self.dropout = torch.nn.Dropout(dropout)
        self.output_layer = torch.nn.Linear(hidden_dim, output_size)
        self.relu = torch.nn.ReLU()
        self.sigmoid = torch.nn.Sigmoid()

    def forward(self, x):
        # Use zip to iterate through both the linear and BN layers together
        for layer, bn in zip(self.layers, self.bn_layers):
            x = layer(x)
            x = bn(x)        # Normalize before activation
            x = self.relu(x)
            x = self.dropout(x)

        x = self.output_layer(x)
        return self.sigmoid(x)

In [9]:
NN = Model(13, n_layers = 2, hidden_dim=32, dropout = .3)

NN.load_state_dict(torch.load("mod13v2.pth"))

<All keys matched successfully>

In [15]:
print(NN)

Model(
  (layers): ModuleList(
    (0): Linear(in_features=13, out_features=32, bias=False)
    (1-2): 2 x Linear(in_features=32, out_features=32, bias=False)
  )
  (bn_layers): ModuleList(
    (0-2): 3 x BatchNorm1d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (dropout): Dropout(p=0.3, inplace=False)
  (output_layer): Linear(in_features=32, out_features=1, bias=True)
  (relu): ReLU()
  (sigmoid): Sigmoid()
)
